<a href="https://colab.research.google.com/github/CipherSunaina02/Flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CipherSunaina02/Flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
# @title AI prompt cell

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown,clear_output
from google.colab import ai

dropdown = widgets.Dropdown(
    options=[],
    layout={'width': 'auto'}
)

def update_model_list(new_options):
    dropdown.options = new_options
update_model_list(ai.list_models())

text_input = widgets.Textarea(
    placeholder='Ask me anything....',
    layout={'width': 'auto', 'height': '100px'},
)

button = widgets.Button(
    description='Submit Text',
    disabled=False,
    tooltip='Click to submit the text',
    icon='check'
)

output_area = widgets.Output(
     layout={'width': 'auto', 'max_height': '300px','overflow_y': 'scroll'}
)

def on_button_clicked(b):
    with output_area:
        output_area.clear_output(wait=False)
        accumulated_content = ""
        for new_chunk in ai.generate_text(prompt=text_input.value, model_name=dropdown.value, stream=True):
            if new_chunk is None:
                continue
            accumulated_content += new_chunk
            clear_output(wait=True)
            display(Markdown(accumulated_content))

button.on_click(on_button_clicked)
vbox = widgets.GridBox([dropdown, text_input, button, output_area])

display(HTML("""
<style>
.widget-dropdown select {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
.widget-textarea textarea {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
</style>
"""))
display(vbox)


GridBox(children=(Dropdown(layout=Layout(width='auto'), options=('google/gemini-2.5-flash', 'google/gemini-2.5…

In [ ]:
import duckdb
import pandas as pd
import numpy as np

from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    "CREATE OR REPLACE SECRET hf (TYPE HUGGINGFACE, TOKEN ?)",
    [HF_TOKEN]
)

FACT = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance"
FEB = f"{FACT}/month=2026-02/*.parquet"

print("Warehouse connected.")
print("Feature window: February 2026")

Warehouse connected.
Feature window: February 2026


In [ ]:
df = con.sql(f"""
SELECT
    content_hash_id,
    SUM(COALESCE(gsc_impressions, 0)) AS impressions,
    SUM(COALESCE(gsc_clicks, 0)) AS clicks,
    AVG(gsc_avg_position) AS avg_position,
    SUM(COALESCE(ga4_pageviews, 0)) AS pageviews,
    SUM(COALESCE(ga4_total_engagement_sec, 0)) AS engagement_sec
FROM read_parquet('{FEB}')
WHERE content_hash_id IS NOT NULL
GROUP BY content_hash_id
""").df()

df["ctr"] = np.where(
    df["impressions"] > 0,
    df["clicks"] / df["impressions"],
    np.nan
)

print("Content items:", len(df))
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Content items: 321546


,content_hash_id,impressions,clicks,avg_position,pageviews,engagement_sec,ctr
0,content_fb84747a57b8b665,0.0,0.0,NaN,0.0,0.0,NaN
1,content_feccf822ac21326e,0.0,0.0,NaN,0.0,0.0,NaN
2,content_17cf93c10413ebe9,0.0,0.0,NaN,0.0,0.0,NaN
3,content_a9905735266f8697,0.0,0.0,NaN,0.0,0.0,NaN
4,content_31c34765e7bba2f0,0.0,0.0,NaN,0.0,0.0,NaN


In [ ]:
position_df = df[
    (df["impressions"] > 0) &
    (df["avg_position"].notna())
].copy()

position_df["position_bucket"] = pd.cut(
    position_df["avg_position"],
    bins=[0, 3, 10, 20, np.inf],
    labels=["1-3", "4-10", "11-20", "20+"]
)

position_check = (
    position_df
    .groupby("position_bucket", observed=False)
    .agg(
        n=("content_hash_id", "count"),
        mean_ctr=("ctr", "mean")
    )
    .reset_index()
)

print(position_check.to_string(index=False))

position_bucket     n  mean_ctr
            1-3 17640  0.008924
           4-10 75898  0.005195
          11-20 31699  0.003060
            20+ 26719  0.002509


In [ ]:
# Signal 2: Impression volume

volume_df = df[
    df["impressions"] > 0
].copy()

volume_df["volume_bucket"] = pd.cut(
    volume_df["impressions"],
    bins=[0, 100, 1000, 10000, np.inf],
    labels=["<100", "100-999", "1K-9,999", "10K+"],
    right=True
)

volume_check = (
    volume_df
    .groupby("volume_bucket", observed=False)
    .agg(
        n=("content_hash_id", "count"),
        mean_ctr=("ctr", "mean"),
        mean_position=("avg_position", "mean")
    )
    .reset_index()
)

print("Signal 2: Impression Volume")
print(volume_check.to_string(index=False))

Signal 2: Impression Volume
volume_bucket     n  mean_ctr  mean_position
         <100 73435  0.007411      14.723334
      100-999 46837  0.002429      12.383713
     1K-9,999 29957  0.003097       8.759707
         10K+  3330  0.003433       8.078404


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### My baseline rule

I will prioritize content for refresh when its observed average organic search position is poor and it has meaningful search visibility. The score will give higher priority to pages with worse average position and more impressions.

**Score:** higher average position + higher search impressions = higher priority.

**Reason code:** `LOW_VISIBILITY_REFRESH`

**Action label:** `REFRESH_CONTENT`

The rule is decision-support only. It does not claim that refreshing a page will cause a ranking improvement.


### Signal check verdicts

**Signal 1 — CTR vs position: CONFIRMED**

In February 2026, mean CTR decreased from 0.008924 for positions 1–3 to 0.002509 for positions 20+. This supports the observed relationship between poorer search position and lower CTR.

**Signal 2 — Impression volume: MIXED**

CTR did not change consistently across impression-volume buckets. It decreased from the `<100` bucket to `100–999`, then increased in the higher-volume buckets. Therefore, impression volume alone is not a reliable directional signal for refresh priority.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-07: Build ranked baseline action queue

queue = df.copy()

# Keep content with measurable organic search visibility
queue = queue[
    (queue["impressions"] > 0) &
    (queue["avg_position"].notna())
].copy()

# Log-transform impressions so extremely high-volume pages do not dominate
queue["volume_score"] = np.log1p(queue["impressions"])

# Normalize position and volume to 0-1
position_min = queue["avg_position"].min()
position_max = queue["avg_position"].max()

volume_min = queue["volume_score"].min()
volume_max = queue["volume_score"].max()

queue["position_score"] = (
    (queue["avg_position"] - position_min) /
    (position_max - position_min)
)

queue["volume_score_norm"] = (
    (queue["volume_score"] - volume_min) /
    (volume_max - volume_min)
)

# Higher score = higher refresh priority
queue["baseline_score"] = (
    0.7 * queue["position_score"] +
    0.3 * queue["volume_score_norm"]
)

queue["reason_code"] = "LOW_VISIBILITY_REFRESH"
queue["action_label"] = "REFRESH_CONTENT"

# Rank highest priority first
queue = queue.sort_values(
    "baseline_score",
    ascending=False
).reset_index(drop=True)

queue["rank"] = np.arange(1, len(queue) + 1)

# Keep the output simple and privacy-safe
output = queue[
    [
        "rank",
        "content_hash_id",
        "baseline_score",
        "reason_code",
        "action_label"
    ]
].copy()

# Create output directory
import os
os.makedirs("work/outputs", exist_ok=True)

# Write required CSV
output.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Ranked queue created.")
print("Rows:", len(output))
print("Output: work/outputs/baseline_action_score.csv")

output.head(10)


Ranked queue created.
Rows: 153559
Output: work/outputs/baseline_action_score.csv


,rank,content_hash_id,baseline_score,reason_code,action_label
0,1,content_af0686c27b91d572,0.700000,LOW_VISIBILITY_REFRESH,REFRESH_CONTENT
1,2,content_039cd00d95fd02a5,0.340600,LOW_VISIBILITY_REFRESH,REFRESH_CONTENT
2,3,content_32b167c958620766,0.337283,LOW_VISIBILITY_REFRESH,REFRESH_CONTENT
3,4,content_ccdf4f5189d9fd8f,0.336177,LOW_VISIBILITY_REFRESH,REFRESH_CONTENT
4,5,content_36e53e9c707674fc,0.319361,LOW_VISIBILITY_REFRESH,REFRESH_CONTENT
5,6,content_8d9e923e3a9805d6,0.315166,LOW_VISIBILITY_REFRESH,REFRESH_CONTENT
6,7,content_e8a52cf3d5988c07,0.309277,LOW_VISIBILITY_REFRESH,REFRESH_CONTENT
7,8,content_532d96d500804552,0.306926,LOW_VISIBILITY_REFRESH,REFRESH_CONTENT
8,9,content_8e1334d6356668e3,0.305493,LOW_VISIBILITY_REFRESH,REFRESH_CONTENT
9,10,content_fec55986a1868d62,0.303014,LOW_VISIBILITY_REFRESH,REFRESH_CONTENT


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-07: Top-20 review

top20 = output.head(20).copy()

top20["confidence_note"] = (
    "Directional priority based on observed February search visibility and impression volume."
)

top20["what_would_make_it_wrong"] = (
    "The page may not need a refresh if the observed position is temporary, "
    "the content is intentionally targeting a low-ranking query, or the underlying "
    "search data is incomplete."
)

top20_review = top20[
    [
        "rank",
        "content_hash_id",
        "action_label",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
]

print("Top-20 review:")
display(top20_review)

Top-20 review:


,rank,content_hash_id,action_label,reason_code,confidence_note,what_would_make_it_wrong
0,1,content_af0686c27b91d572,REFRESH_CONTENT,LOW_VISIBILITY_REFRESH,Directional priority based on observed Februar...,The page may not need a refresh if the observe...
1,2,content_039cd00d95fd02a5,REFRESH_CONTENT,LOW_VISIBILITY_REFRESH,Directional priority based on observed Februar...,The page may not need a refresh if the observe...
2,3,content_32b167c958620766,REFRESH_CONTENT,LOW_VISIBILITY_REFRESH,Directional priority based on observed Februar...,The page may not need a refresh if the observe...
3,4,content_ccdf4f5189d9fd8f,REFRESH_CONTENT,LOW_VISIBILITY_REFRESH,Directional priority based on observed Februar...,The page may not need a refresh if the observe...
4,5,content_36e53e9c707674fc,REFRESH_CONTENT,LOW_VISIBILITY_REFRESH,Directional priority based on observed Februar...,The page may not need a refresh if the observe...
5,6,content_8d9e923e3a9805d6,REFRESH_CONTENT,LOW_VISIBILITY_REFRESH,Directional priority based on observed Februar...,The page may not need a refresh if the observe...
6,7,content_e8a52cf3d5988c07,REFRESH_CONTENT,LOW_VISIBILITY_REFRESH,Directional priority based on observed Februar...,The page may not need a refresh if the observe...
7,8,content_532d96d500804552,REFRESH_CONTENT,LOW_VISIBILITY_REFRESH,Directional priority based on observed Februar...,The page may not need a refresh if the observe...
8,9,content_8e1334d6356668e3,REFRESH_CONTENT,LOW_VISIBILITY_REFRESH,Directional priority based on observed Februar...,The page may not need a refresh if the observe...
9,10,content_fec55986a1868d62,REFRESH_CONTENT,LOW_VISIBILITY_REFRESH,Directional priority based on observed Februar...,The page may not need a refresh if the observe...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*



The weakest picks are the lower-ranked rows near the bottom of the queue. They receive lower baseline scores because they have lower combined refresh priority under this rule. They should not automatically be treated as poor content; they are only lower-priority candidates.

The baseline uses only February 2026 observed search signals: impressions and average search position. I did not use future-month data, a label-derived field, or product flags. Therefore the baseline does not use future outcomes to rank the queue.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-07: Weak picks + leakage check

print("=== Weak Picks ===")

# Review a few lower-ranked candidates near the decision boundary
weak_picks = output.tail(10).copy()

display(
    weak_picks[
        [
            "rank",
            "content_hash_id",
            "baseline_score",
            "reason_code",
            "action_label"
        ]
    ]
)

print("\n=== Leakage Check ===")

# Features used by the baseline
features_used = [
    "impressions",
    "avg_position"
]

print("Features used:", features_used)

# Confirm that no future-month fields or label-derived fields are used
forbidden_terms = [
    "trend",
    "label",
    "future",
    "outcome",
    "march",
    "product"
]

leakage_columns = [
    col for col in features_used
    if any(term in col.lower() for term in forbidden_terms)
]

print("Potential leakage columns:", leakage_columns)

if len(leakage_columns) == 0:
    print("LEAKAGE CHECK: PASSED")
else:
    print("LEAKAGE CHECK: REVIEW REQUIRED")


=== Weak Picks ===


,rank,content_hash_id,baseline_score,reason_code,action_label
153549,153550,content_6461cfce4381b4a6,0.0,LOW_VISIBILITY_REFRESH,REFRESH_CONTENT
153550,153551,content_75d72e6e073de022,0.0,LOW_VISIBILITY_REFRESH,REFRESH_CONTENT
153551,153552,content_fba0ab526737f2d3,0.0,LOW_VISIBILITY_REFRESH,REFRESH_CONTENT
153552,153553,content_a4aa2347feeb92cf,0.0,LOW_VISIBILITY_REFRESH,REFRESH_CONTENT
153553,153554,content_c526bc6a1f1f3f1f,0.0,LOW_VISIBILITY_REFRESH,REFRESH_CONTENT
153554,153555,content_a3ccc05d51e7dd76,0.0,LOW_VISIBILITY_REFRESH,REFRESH_CONTENT
153555,153556,content_d45650ab162e6f32,0.0,LOW_VISIBILITY_REFRESH,REFRESH_CONTENT
153556,153557,content_14e9761bea6c6b2d,0.0,LOW_VISIBILITY_REFRESH,REFRESH_CONTENT
153557,153558,content_e03a7f4bb6fa29a8,0.0,LOW_VISIBILITY_REFRESH,REFRESH_CONTENT
153558,153559,content_ce208a5082b4732e,0.0,LOW_VISIBILITY_REFRESH,REFRESH_CONTENT



=== Leakage Check ===
Features used: ['impressions', 'avg_position']
Potential leakage columns: []
LEAKAGE CHECK: PASSED


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.